In [14]:
import os
import json
import torch
import random
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torchaudio
import torchaudio.transforms as T
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, precision_recall_curve, roc_auc_score, average_precision_score
from tqdm import tqdm
import shutil
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [15]:
def global_mel_stats(flac_dir, mel_transform, amplitube_to_db, max_len=16000*4, sample_limit=2000):
    all_vals=[]
    files=os.listdir(flac_dir)
    if len(files)>sample_limit:
        files=random.sample(files,sample_limit)
    for file in tqdm(files,desc="Computing mel stats"):
        waveform,sr=torchaudio.load(os.path.join(flac_dir,file))
        if waveform.shape[0]>1:
            waveform=waveform.mean(dim=0,keepdim=True)
        if sr!=16000:
            waveform=T.Resample(sr,16000)(waveform)
        if waveform.shape[1]>max_len:
            waveform=waveform[:, :max_len]
        else:
            waveform=torch.nn.functional.pad(waveform,(0,max_len-waveform.shape[1]))
        mel=mel_transform(waveform)
        mel=amplitude_to_db(mel)
        all_vals.append(mel)
    all_vals=torch.cat(all_vals,dim=0)
    return all_vals.mean().item(), all_vals.std().item()

In [16]:
mel_transform = T.MelSpectrogram(sample_rate=16000, n_fft=1024, hop_length=256, n_mels=128)
amplitude_to_db = T.AmplitudeToDB()
max_len = 16000 * 4

train_flac_dir = "/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_train/flac"
train_mel_dir = "/kaggle/working/mel_train"
dev_flac_dir="/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_dev/flac"
dev_mel_dir="/kaggle/working/mel_dev"
eval_flac_dir="/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_eval/flac"
eval_mel_dir="/kaggle/working/mel_eval"

os.makedirs(train_mel_dir, exist_ok=True)
os.makedirs(dev_mel_dir, exist_ok=True)

def compute_global_stats(flac_dir, sample_limit=2000):
    mel_vals = []
    delta_vals = []
    delta2_vals = []

    files = os.listdir(flac_dir)
    if len(files) > sample_limit:
        import random
        files = random.sample(files, sample_limit)

    for file in tqdm(files, desc="Computing global stats"):
        waveform, sr = torchaudio.load(os.path.join(flac_dir, file))

        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        if sr != 16000:
            waveform = T.Resample(sr, 16000)(waveform)

        if waveform.shape[1] > max_len:
            waveform = waveform[:, :max_len]
        else:
            waveform = torch.nn.functional.pad(
                waveform, (0, max_len - waveform.shape[1])
            )

        mel = mel_transform(waveform)
        mel = amplitude_to_db(mel)

        delta = torchaudio.functional.compute_deltas(mel)
        delta2 = torchaudio.functional.compute_deltas(delta)

        mel_vals.append(mel)
        delta_vals.append(delta)
        delta2_vals.append(delta2)

    mel_vals = torch.cat(mel_vals, dim=0)
    delta_vals = torch.cat(delta_vals, dim=0)
    delta2_vals = torch.cat(delta2_vals, dim=0)

    return (
        mel_vals.mean().item(), mel_vals.std().item(),
        delta_vals.mean().item(), delta_vals.std().item(),
        delta2_vals.mean().item(), delta2_vals.std().item()
    )


mel_mean, mel_std, delta_mean, delta_std, delta2_mean, delta2_std = compute_global_stats(train_flac_dir)

print(f"Mel mean: {mel_mean:.4f}, std: {mel_std:.4f}")
print(f"Delta mean: {delta_mean:.4f}, std: {delta_std:.4f}")
print(f"Delta2 mean: {delta2_mean:.4f}, std: {delta2_std:.4f}")

Computing global stats: 100%|██████████| 2000/2000 [00:39<00:00, 50.29it/s]


Mel     mean: -38.0290, std: 38.5913
Delta   mean: -0.1480, std: 3.9119
Delta2  mean: -0.0000, std: 1.6467


In [ ]:
mel_transform=T.MelSpectrogram(sample_rate=16000,n_fft=1024,hop_length=256,n_mels=128)
amplitude_to_db=T.AmplitudeToDB()
max_len=16000*4
train_flac_dir="/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_train/flac"
dev_flac_dir="/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_dev/flac"
#eval_flac_dir="/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_eval/flac"

train_mel_dir="/kaggle/working/mel_train"
dev_mel_dir="/kaggle/working/mel_dev"
#eval_mel_dir="/kaggle/working/mel_eval"

for d in [dev_mel_dir]:
    os.makedirs(d, exist_ok=True)

def process_and_save(flac_dir, out_dir, global_mean, global_std):
    for file in tqdm(os.listdir(flac_dir), desc=f"Processing {os.path.basename(flac_dir)}"):
        out_path=os.path.join(out_dir,file.replace(".flac",".pt"))
        if os.path.exists(out_path):
            continue
        waveform,sr=torchaudio.load(os.path.join(flac_dir,file))
        if waveform.shape[0]>1:
            waveform=waveform.mean(dim=0, keepdim=True)
        if sr!=16000:
            waveform=T.Resample(sr, 16000)(waveform)
        if waveform.shape[1]>max_len:
            waveform=waveform[:, :max_len]
        else:
            waveform=torch.nn.functional.pad(waveform, (0, max_len - waveform.shape[1]))
        mel=mel_transform(waveform)
        mel=amplitude_to_db(mel)
        delta=torchaudio.functional.compute_deltas(mel)
        delta2=torchaudio.functional.compute_deltas(delta)

        mel=(mel-global_mean)/(mel_std+1e-6)
        delta=(delta-global_mean)/(delta_std+1e-6)
        delta2=(delta2-global_mean)/(delta2_std+1e-6)

        mel_3ch=torch.cat([mel,delta,delta2],dim=0)
        torch.save(mel_3ch,out_path)

process_and_save(dev_flac_dir,dev_mel_dir,global_mean,global_std)
shutil.make_archive("/kaggle/working/mel_dev", 'zip', "/kaggle/working/mel_dev")

In [ ]:
shutil.rmtree("/kaggle/working/mel_dev", ignore_errors=True)

In [18]:
def rawboost_augment(waveform):
    snr_db=random.uniform(15, 40)
    sig_pow=waveform.pow(2).mean()
    noise=torch.randn_like(waveform)
    noise_pow=noise.pow(2).mean()
    scale=torch.sqrt(sig_pow / (noise_pow * 10 ** (snr_db / 10) + 1e-8))
    return waveform + scale * noise

In [19]:
class ASVspoofDataset(Dataset):
    def __init__(self, protocol_path, mel_dir=None, flac_dir=None, augment=False):
        self.mel_dir = mel_dir
        self.flac_dir = flac_dir
        self.augment = augment
        self.samples = []

        with open(protocol_path, "r") as f:
            for line in f.readlines():
                parts = line.strip().split()
                filename = parts[1]
                label = 0 if parts[4] == "bonafide" else 1
                self.samples.append((filename, label))

    def __len__(self):
        return len(self.samples)

    def _process_audio(self, filepath):
        waveform, sr = torchaudio.load(filepath)

        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        if sr != 16000:
            waveform = T.Resample(sr, 16000)(waveform)

        if waveform.shape[1] > max_len:
            waveform = waveform[:, :max_len]
        else:
            waveform = torch.nn.functional.pad(
                waveform, (0, max_len - waveform.shape[1])
            )
        if self.augment:
            waveform = rawboost_augment(waveform)
        mel = mel_transform(waveform)
        mel = amplitude_to_db(mel)
        delta = torchaudio.functional.compute_deltas(mel)
        delta2 = torchaudio.functional.compute_deltas(delta)
        mel = (mel - mel_mean) / (mel_std + 1e-6)
        delta = (delta - delta_mean) / (delta_std + 1e-6)
        delta2 = (delta2 - delta2_mean) / (delta2_std + 1e-6)

        return torch.cat([mel, delta, delta2], dim=0)

    def _spec_augment(self, mel):
        C, F, T = mel.shape
        mel = mel.clone()

        f_mask = random.randint(0, 20)
        f0 = random.randint(0, F - f_mask - 1)
        mel[0, f0:f0 + f_mask, :] = 0

        t_mask = random.randint(0, 40)
        t0 = random.randint(0, T - t_mask - 1)
        mel[:, :, t0:t0 + t_mask] = 0

        return mel

    def __getitem__(self, idx):
        filename, label = self.samples[idx]

        if self.mel_dir:
            path = os.path.join(self.mel_dir, filename + ".pt")

            if not os.path.exists(path):
                raise FileNotFoundError(f"Missing mel file: {path}")

            mel = torch.load(path)
            if self.augment:
                mel = self._spec_augment(mel)
        else:
            mel = self._process_audio(os.path.join(self.flac_dir, filename + ".flac"))

        if self.augment:
            mel = self._spec_augment(mel)

        return mel.float(), torch.tensor(label, dtype=torch.float32)

In [20]:
train_protocol="/kaggle/input/datasets/rudranshiverma/asvspoof-protocols/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"
dev_protocol="/kaggle/input/datasets/rudranshiverma/asvspoof-protocols/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt"
eval_protocol="/kaggle/input/datasets/rudranshiverma/asvspoof-protocols/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt"

train_dataset = ASVspoofDataset(
    train_protocol,
    mel_dir="/kaggle/input/datasets/rudranshiverma/train-mel-v2",
    augment=True
)

dev_dataset = ASVspoofDataset(
    dev_protocol,
    mel_dir="/kaggle/input/datasets/rudranshiverma/dev-mel-v2",
    augment=False
)

eval_dataset = ASVspoofDataset(
    eval_protocol,
    flac_dir=eval_flac_dir,
    augment=False
)

In [21]:
train_loader=DataLoader(train_dataset,batch_size=64,shuffle=True,num_workers=2,pin_memory=True,persistent_workers=True)
dev_loader=DataLoader(dev_dataset,batch_size=64,shuffle=False,num_workers=2,pin_memory=True,persistent_workers=True)
eval_loader=DataLoader(eval_dataset,batch_size=64,shuffle=False,num_workers=2,pin_memory=True)
print(f"Train batches: {len(train_loader)}")

Train batches: 397


In [22]:
n_bonafide=sum(1 for _, l in train_dataset.samples if l==0)
n_spoof=sum(1 for _, l in train_dataset.samples if l==1)
print(f"Train samples: {len(train_dataset.samples)}")
print(f"Bonafide: {n_bonafide} | Spoof: {n_spoof} | Ratio: {n_spoof/n_bonafide:.2f}:1")

Train samples: 25380
Bonafide: 2580 | Spoof: 22800 | Ratio: 8.84:1


In [23]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
 
    def forward(self, logits, targets):
        bce=nn.functional.binary_cross_entropy_with_logits(
                   logits, targets, reduction='none')
        pt=torch.exp(-bce)
        at=self.alpha * targets + (1-self.alpha)*(1-targets)
        loss = at*(1-pt)**self.gamma * bce
        return loss.mean()

In [24]:
def build_model():
    model=models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for name,param in model.named_parameters():
        if "layer2" not in name and "layer3" not in name and "layer4" not in name:
            param.requires_grad=False
    
    in_feat=model.fc.in_features
    model.fc=nn.Sequential(
        nn.BatchNorm1d(in_feat),
        nn.Dropout(0.3),
        nn.Linear(in_feat,1)
    )
    return model.to(device)
model=build_model()
print(model.fc)
criterion = FocalLoss(alpha=0.75, gamma=2.0)
optimizer=optim.AdamW(model.parameters(),lr=1e-4,weight_decay=1e-4)

Sequential(
  (0): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (1): Dropout(p=0.3, inplace=False)
  (2): Linear(in_features=512, out_features=1, bias=True)
)


In [25]:
scheduler=optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,mode='min',factor=0.5,patience=3
)
epochs=20
patience=5
min_epochs=8
epochs_no_improve=0
best_pr_auc       = -1.0   
best_eer=float("inf")
history={
    "train_loss": [], "val_loss": [], "val_accuracy": [],
    "val_precision": [], "val_recall": [], "val_f1": [], "val_eer": [], "val_roc_auc": [], "val_pr_auc": []
}
for epoch in range(epochs):
    model.train()
    running_loss=0.0
    for inputs,labels in train_loader:
        inputs=inputs.to(device)
        labels=labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        outputs=model(inputs)
        loss=criterion(outputs,labels)
        loss.backward()
        optimizer.step()
        running_loss+=loss.item()
    avg_train_loss=running_loss/len(train_loader)

    model.eval()
    val_loss=0.0
    all_labels=[]
    all_scores=[]
    with torch.no_grad():
        for inputs,labels in dev_loader:
            inputs=inputs.to(device)
            labels=labels.to(device).unsqueeze(1)
            outputs=model(inputs)
            loss=criterion(outputs,labels)
            val_loss+=loss.item()
            scores=torch.sigmoid(outputs).view(-1)
            all_scores.extend(scores.cpu().numpy())
            all_labels.extend(labels.view(-1).cpu().numpy())
    avg_val_loss=val_loss/len(dev_loader)
    all_scores=np.array(all_scores)
    all_labels=np.array(all_labels)

    precision_arr, recall_arr, thresholds_pr = precision_recall_curve(all_labels, all_scores)
    
    valid = np.where(recall_arr[:-1] >= 0.90)[0]
    if len(valid) > 0:
        best_idx         = valid[np.argmax(precision_arr[valid])]
        recall_threshold = thresholds_pr[best_idx]
    else:
        #fallback to max-F1 with warning instead of arbitrary 0.5
        f1_scores=(2 * precision_arr[:-1] * recall_arr[:-1]/(precision_arr[:-1] + recall_arr[:-1] + 1e-8))
        best_idx= np.argmax(f1_scores)
        recall_threshold = thresholds_pr[best_idx]
        print(f"  WARNING: recall >= 0.90 not achievable at any threshold. "
              f"Falling back to max-F1 threshold ({recall_threshold:.4f}).")
    
    
    fpr,tpr,thresholds_roc=roc_curve(all_labels,all_scores)
    fnr=1-tpr
    eer_index=np.nanargmin(np.abs(fnr-fpr))
    eer=fpr[eer_index]
    eer_threshold=thresholds_roc[eer_index]
    
    preds=(all_scores > recall_threshold).astype(int)
    accuracy=accuracy_score(all_labels, preds)
    precision=precision_score(all_labels, preds, zero_division=0)
    recall=recall_score(all_labels, preds, zero_division=0)
    f1=f1_score(all_labels, preds, zero_division=0)
    cm=confusion_matrix(all_labels, preds)
    roc_auc = roc_auc_score(all_labels, all_scores)
    pr_auc = average_precision_score(all_labels, all_scores)
    scheduler.step(avg_val_loss)

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["val_accuracy"].append(accuracy)
    history["val_precision"].append(precision)
    history["val_recall"].append(recall)
    history["val_f1"].append(f1)
    history["val_eer"].append(eer)
    history.setdefault("val_roc_auc", []).append(roc_auc)
    history.setdefault("val_pr_auc", []).append(pr_auc)

    print(f"\nEpoch {epoch+1}/{epochs}")
    print(f"Train Loss:{avg_train_loss:.6f}")
    print(f"Val Loss:{avg_val_loss:.6f}")
    print(f"EER: {eer:.4f}")
    print(f"EER Threshold: {eer_threshold:.4f}")
    print(f"Recall Threshold: {recall_threshold:.4f}")
    print(f"Accuracy: {accuracy:.4f} Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f} F1: {f1:.4f}")
    print(f"  ROC-AUC          : {roc_auc:.4f}  PR-AUC: {pr_auc:.4f}")
    print(f"Score dist — min:{all_scores.min():.3f} max:{all_scores.max():.3f} "
          f"mean:{all_scores.mean():.3f} std:{all_scores.std():.3f}")
    print(f"Confusion Matrix:\n{cm}")

    if pr_auc > best_pr_auc:
        best_pr_auc = pr_auc
        best_threshold=recall_threshold
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "eer": eer,
            "eer_threshold": eer_threshold,
            "recall_threshold": recall_threshold,
            "pr_auc": pr_auc,
            "roc_auc": roc_auc,
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "confusion_matrix": cm.tolist(),
            "mel_mean": mel_mean,
            "mel_std": mel_std,
            "delta_mean": delta_mean,
            "delta_std": delta_std,
            "delta2_mean": delta2_mean,
            "delta2_std": delta2_std,
        }, "/kaggle/working/audio_resnet18.pth")
        print("  ✓ Best model saved")
        epochs_no_improve = 0
    else:
        if epoch >= min_epochs:
            epochs_no_improve += 1
            print(f"No improvement {epochs_no_improve}/{patience}")
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}. "
                      f"Best dev PR-AUC: {best_pr_auc:.4f}")
                break
with open("/kaggle/working/audio_resnet18_history.json", "w") as f:
    json.dump(history, f)
print("\nTraining complete.")
print(f"Best dev EER : {best_eer:.4f}")
print(f"Best dev PR-AUC : {best_pr_auc:.4f}")
print(f"Best threshold (from dev): {best_threshold:.4f}")


Epoch 1/20
Train Loss:0.090128
Val Loss:0.016844
EER: 0.0718
EER Threshold: 0.6660
Recall Threshold: 0.6751
Accuracy: 0.9077 Precision: 0.9915
Recall: 0.9050 F1: 0.9463
  ROC-AUC          : 0.9655  PR-AUC: 0.9923
Score dist — min:0.000 max:0.920 mean:0.686 std:0.214
Confusion Matrix:
[[ 2375   173]
 [ 2119 20177]]
  ✓ Best model saved

Epoch 2/20
Train Loss:0.021942
Val Loss:0.006860
EER: 0.0271
EER Threshold: 0.8024
Recall Threshold: 0.8327
Accuracy: 0.9106 Precision: 0.9988
Recall: 0.9015 F1: 0.9477
  ROC-AUC          : 0.9955  PR-AUC: 0.9994
Score dist — min:0.000 max:0.950 mean:0.807 std:0.222
Confusion Matrix:
[[ 2524    24]
 [ 2196 20100]]
  ✓ Best model saved

Epoch 3/20
Train Loss:0.009658
Val Loss:0.001913
EER: 0.0059
EER Threshold: 0.7912
Recall Threshold: 0.8625
Accuracy: 0.9173 Precision: 0.9999
Recall: 0.9080 F1: 0.9517
  ROC-AUC          : 0.9996  PR-AUC: 0.9999
Score dist — min:0.000 max:0.967 mean:0.814 std:0.260
Confusion Matrix:
[[ 2545     3]
 [ 2051 20245]]
  ✓ Bes

In [26]:
checkpoint = torch.load("/kaggle/working/audio_resnet18.pth",
                        map_location=device, weights_only=False)
model = build_model()
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

all_labels, all_scores = [], []
with torch.no_grad():
    for inputs, labels in dev_loader:
        inputs  = inputs.to(device)
        outputs = model(inputs)
        scores  = torch.sigmoid(outputs).view(-1)
        all_scores.extend(scores.cpu().numpy())
        all_labels.extend(labels.view(-1).cpu().numpy())

all_scores = np.array(all_scores)
all_labels = np.array(all_labels)

#sweeping thresholds to find best recall where precision >= 0.99
print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Missed Spoof':>14}")
print("-" * 60)

best_threshold = None
best_recall    = 0.0

for thresh in np.arange(0.30, 0.96, 0.01):
    preds= (all_scores > thresh).astype(int)
    prec= precision_score(all_labels, preds, zero_division=0)
    rec= recall_score(all_labels, preds, zero_division=0)
    f1= f1_score(all_labels, preds, zero_division=0)
    cm= confusion_matrix(all_labels, preds)
    missed= cm[1][0]
    print(f"{thresh:>10.2f} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f} {missed:>14}")
    if prec >= 0.99 and rec > best_recall:
        best_recall= rec
        best_threshold = thresh

print(f"\nBest threshold on dev: {best_threshold:.2f}")
print(f"Best recall at precision>=0.99: {best_recall:.4f}")

 Threshold  Precision     Recall         F1   Missed Spoof
------------------------------------------------------------
      0.30     0.9887     1.0000     0.9943              0
      0.31     0.9890     1.0000     0.9944              0
      0.32     0.9893     1.0000     0.9946              0
      0.33     0.9895     1.0000     0.9947              0
      0.34     0.9898     1.0000     0.9949              0
      0.35     0.9900     1.0000     0.9950              0
      0.36     0.9903     1.0000     0.9951              0
      0.37     0.9904     1.0000     0.9952              0
      0.38     0.9906     1.0000     0.9953              0
      0.39     0.9908     1.0000     0.9954              0
      0.40     0.9913     1.0000     0.9956              0
      0.41     0.9914     1.0000     0.9957              0
      0.42     0.9918     1.0000     0.9959              0
      0.43     0.9923     1.0000     0.9961              0
      0.44     0.9925     1.0000     0.9962           

In [31]:
checkpoint = torch.load("/kaggle/working/audio_resnet18.pth",
                        map_location=device, weights_only=False)
model=build_model()
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()
eval_threshold = 0.45
all_labels=[]
all_scores=[]
with torch.no_grad():
    for inputs,labels in eval_loader:
        inputs=inputs.to(device)
        outputs=model(inputs)
        scores=torch.sigmoid(outputs).view(-1)
        all_scores.extend(scores.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
all_scores=np.array(all_scores)
all_labels=np.array(all_labels)
preds=(all_scores > eval_threshold).astype(int)
accuracy=accuracy_score(all_labels, preds)
precision = precision_score(all_labels, preds, zero_division=0)
recall=recall_score(all_labels, preds, zero_division=0)
f1=f1_score(all_labels, preds, zero_division=0)
cm=confusion_matrix(all_labels, preds)
fpr,tpr, _=roc_curve(all_labels, all_scores)
fnr=1-tpr
eer=fpr[np.nanargmin(np.abs(fnr-fpr))]
roc_auc = roc_auc_score(all_labels, all_scores)
pr_auc = average_precision_score(all_labels, all_scores)

print("\nEval Results")
print(f"EER: {eer:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1: {f1:.4f}")
print(f"Threshold: {eval_threshold:.4f} (from dev set)")
print(f"Confusion Matrix:\n{cm}")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")
print(f"Score dist — min:{all_scores.min():.3f} max:{all_scores.max():.3f} "
      f"mean:{all_scores.mean():.3f} std:{all_scores.std():.3f}")
eval_metrics = {
    "accuracy":float(accuracy),
    "precision":float(precision),
    "recall":float(recall),
    "f1":float(f1),
    "eer":float(eer),
    "threshold_used":float(eval_threshold),
    "roc_auc": float(roc_auc),
    "pr_auc": float(pr_auc),
    "threshold_source":"dev_sweep_selected",
    "confusion_matrix":cm.tolist(),
}
with open("/kaggle/working/audio_resnet18_eval_metrics.json", "w") as f:
    json.dump(eval_metrics, f, indent=4)
 
np.save("/kaggle/working/audio_resnet18_fpr.npy", fpr)
np.save("/kaggle/working/audio_resnet18_tpr.npy", tpr)
np.save("/kaggle/working/resnet_scores.npy", all_scores)
np.save("/kaggle/working/resnet_labels.npy", all_labels)
print("\nEval metrics saved.")


Eval Results
EER: 0.0662
Accuracy: 0.9240
Precision: 0.9947
Recall: 0.9201
F1: 0.9560
Threshold: 0.4500 (from dev set)
Confusion Matrix:
[[ 7041   314]
 [ 5101 58781]]
ROC-AUC: 0.9828
PR-AUC: 0.9980
Score dist — min:0.000 max:0.981 mean:0.759 std:0.326

Eval metrics saved.
